# Possession vs Directness Analysis

This notebook analyzes whether teams with higher possession also showed stronger direct progression during the 2026 FIFA World Cup.

The analysis uses team-level FIFA Match Centre data and focuses on the relationship between possession, passing volume, ball progression, line breaks, and final-third entries.

## Main Question

Did higher possession translate into more direct attacking progression?

## Key Ideas

- Possession measures how much control a team had over the ball.
- Directness measures how effectively that control was turned into forward progression.
- A team can have high possession but low directness.
- A team can also have lower possession but progress quickly and directly.

## Data Source

FIFA Match Centre, full FIFA Official Stats only.

Belgium vs Egypt is excluded from full-stat analysis because FIFA provides only Live Statistics for that match, not the full official stats section.

## Important Notes

- Rankings are descriptive, not causal.
- Match counts differ from 3 to 8, so smaller samples may be more volatile.
- xG and opponent strength are not controlled.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Optional label adjustment library
try:
    from adjustText import adjust_text
    ADJUST_TEXT_AVAILABLE = True
except ImportError:
    ADJUST_TEXT_AVAILABLE = False

# =========================================================
# Edit only this line if the project folder is moved
# =========================================================
BASE_DIR = Path.cwd().resolve()
for parent in [BASE_DIR, *BASE_DIR.parents]:
    if parent.name == "worldcup-2026-official-stats-analysis":
        BASE_DIR = parent
        break

DATA_DIR = BASE_DIR / "data" / "fifa_worldcup_2026" / "site_scrape"
RAW_CSV_PATH = DATA_DIR / "site_official_stats_team_wide_flagged.csv"

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Input file:", RAW_CSV_PATH)
print("Output directory:", OUTPUT_DIR)
print("adjustText available:", ADJUST_TEXT_AVAILABLE)

In [ ]:
# =========================================================
# Step 2. Load the FIFA official stats dataset
# =========================================================

raw = pd.read_csv(RAW_CSV_PATH)

print("Raw dataset shape:", raw.shape)
print("Number of match-team rows:", len(raw))
print("Number of matches:", raw["match_id"].nunique())
print("Number of teams:", raw["team_name"].nunique())

display(raw.head())

In [ ]:
 # =========================================================
# Step 3. Check missing values and full-stat availability
# =========================================================

# Columns needed for Possession vs Directness analysis
analysis_columns = [
    "match_id",
    "team_name",
    "opponent_name",
    "stats_complete",
    "stats_source",
    "attacking__possession",
    "attacking__distribution__passes",
    "attacking__distribution__passes_completed",
    "attacking__line_breaks__attempted_line_breaks",
    "attacking__line_breaks__completed_line_breaks",
    "attacking__line_breaks__attempted_defensive_line_breaks",
    "attacking__line_breaks__completed_defensive_line_breaks",
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

missing_columns = [col for col in analysis_columns if col not in raw.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available.")

In [ ]:
# =========================================================
# Step 3A. Separate full official stats from incomplete stats
# =========================================================

full_stats = raw[raw["stats_complete"] == True].copy()
incomplete_stats = raw[raw["stats_complete"] != True].copy()

print("Rows in raw data:", len(raw))
print("Rows with full FIFA Official Stats:", len(full_stats))
print("Rows without full FIFA Official Stats:", len(incomplete_stats))

print("Matches in raw data:", raw["match_id"].nunique())
print("Matches with full FIFA Official Stats:", full_stats["match_id"].nunique())
print("Teams in full FIFA Official Stats:", full_stats["team_name"].nunique())

if len(incomplete_stats) > 0:
    print("\nRows excluded from full-stat analysis:")
    display(
        incomplete_stats[
            [
                "match_id",
                "team_name",
                "opponent_name",
                "stats_complete",
                "stats_source",
            ]
        ]
    )

In [ ]:
# =========================================================
# Step 3B. Inspect Belgium vs Egypt rows directly
# =========================================================

belgium_egypt_check = raw[
    raw["team_name"].isin(["Belgium", "Egypt"])
    & raw["opponent_name"].isin(["Belgium", "Egypt"])
].copy()

display(
    belgium_egypt_check[
        [
            "match_id",
            "team_name",
            "opponent_name",
            "stats_complete",
            "stats_source",
            "attacking__possession",
            "attacking__distribution__passes_completed",
            "attacking__line_breaks__completed_line_breaks",
        ]
    ]
)

In [ ]:
# =========================================================
# Step 3C. Check missing values in full-stat rows
# =========================================================

metric_columns = [
    "attacking__possession",
    "attacking__distribution__passes",
    "attacking__distribution__passes_completed",
    "attacking__line_breaks__attempted_line_breaks",
    "attacking__line_breaks__completed_line_breaks",
    "attacking__line_breaks__attempted_defensive_line_breaks",
    "attacking__line_breaks__completed_defensive_line_breaks",
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

missing_summary = (
    full_stats[metric_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]

display(missing_summary)

In [ ]:
# =========================================================
# Step 3D. Check suspicious decimal values in count-based columns
# =========================================================

count_columns = [
    "attacking__distribution__passes",
    "attacking__distribution__passes_completed",
    "attacking__line_breaks__attempted_line_breaks",
    "attacking__line_breaks__completed_line_breaks",
    "attacking__line_breaks__attempted_defensive_line_breaks",
    "attacking__line_breaks__completed_defensive_line_breaks",
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

decimal_issues = []

for col in count_columns:
    decimal_rows = full_stats[
        full_stats[col].notna()
        & (full_stats[col] % 1 != 0)
    ][["match_id", "team_name", col]]
    
    if len(decimal_rows) > 0:
        decimal_issues.append((col, decimal_rows))

if decimal_issues:
    for col, rows in decimal_issues:
        print(f"\nDecimal values found in: {col}")
        display(rows)
else:
    print("No suspicious decimal values found in count-based columns.")

In [ ]:
 # =========================================================
# Step 4. Clean team possession and create match-team metrics
# =========================================================

analysis_df = full_stats.copy()

# Helper function:
# Extract the first numeric value from text.
# Examples:
# "52%" -> 52
# "8 in contest" -> 8
# NaN -> NaN
def extract_first_number(series):
    return (
        series
        .astype(str)
        .str.extract(r"(\d+\.?\d*)")[0]
        .astype(float)
    )

# Convert FIFA team possession from strings like "52%" into numeric values.
analysis_df["team_possession_pct"] = extract_first_number(
    analysis_df["attacking__possession"]
)

# Keep in-contest possession as a separate reference column.
# It is not allocated to either team.
analysis_df["in_contest_pct"] = extract_first_number(
    analysis_df["attacking__possession__in_contest"]
)

# FIFA final-third entries are split into five attacking channels.
final_third_entry_columns = [
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

analysis_df["final_third_entries"] = analysis_df[final_third_entry_columns].sum(axis=1, min_count=5)

# Pass completion rate
analysis_df["pass_completion_rate"] = np.where(
    analysis_df["attacking__distribution__passes"] > 0,
    analysis_df["attacking__distribution__passes_completed"]
    / analysis_df["attacking__distribution__passes"]
    * 100,
    np.nan,
)

# Line break completion rate
analysis_df["line_break_completion_rate"] = np.where(
    analysis_df["attacking__line_breaks__attempted_line_breaks"] > 0,
    analysis_df["attacking__line_breaks__completed_line_breaks"]
    / analysis_df["attacking__line_breaks__attempted_line_breaks"]
    * 100,
    np.nan,
)

# Defensive line break completion rate
analysis_df["defensive_line_break_completion_rate"] = np.where(
    analysis_df["attacking__line_breaks__attempted_defensive_line_breaks"] > 0,
    analysis_df["attacking__line_breaks__completed_defensive_line_breaks"]
    / analysis_df["attacking__line_breaks__attempted_defensive_line_breaks"]
    * 100,
    np.nan,
)

# Directness proxy 1:
# How often a team reached the final third relative to completed passes.
analysis_df["final_third_entries_per_100_completed_passes"] = np.where(
    analysis_df["attacking__distribution__passes_completed"] > 0,
    analysis_df["final_third_entries"]
    / analysis_df["attacking__distribution__passes_completed"]
    * 100,
    np.nan,
)

# Directness proxy 2:
# How often a team completed line breaks relative to completed passes.
analysis_df["completed_line_breaks_per_100_completed_passes"] = np.where(
    analysis_df["attacking__distribution__passes_completed"] > 0,
    analysis_df["attacking__line_breaks__completed_line_breaks"]
    / analysis_df["attacking__distribution__passes_completed"]
    * 100,
    np.nan,
)

# Display a cleaner preview table.
# The underlying analysis_df keeps full precision for later calculations.
preview_columns = [
    "match_id",
    "team_name",
    "opponent_name",
    "team_possession_pct",
    "in_contest_pct",
    "attacking__distribution__passes_completed",
    "final_third_entries",
    "attacking__line_breaks__completed_line_breaks",
    "pass_completion_rate",
    "line_break_completion_rate",
    "defensive_line_break_completion_rate",
    "final_third_entries_per_100_completed_passes",
    "completed_line_breaks_per_100_completed_passes",
]

display(
    analysis_df[preview_columns]
    .head()
    .round(2)
)

In [ ]:
# =========================================================
# Step 5. Aggregate match-team data to team-level metrics
# =========================================================

team_metrics = (
    analysis_df
    .groupby("team_name", as_index=False)
    .agg(
        matches_played=("match_id", "nunique"),
        avg_team_possession_pct=("team_possession_pct", "mean"),
        avg_in_contest_pct=("in_contest_pct", "mean"),
        total_passes=("attacking__distribution__passes", "sum"),
        total_completed_passes=("attacking__distribution__passes_completed", "sum"),
        total_final_third_entries=("final_third_entries", "sum"),
        total_attempted_line_breaks=("attacking__line_breaks__attempted_line_breaks", "sum"),
        total_completed_line_breaks=("attacking__line_breaks__completed_line_breaks", "sum"),
        total_attempted_defensive_line_breaks=("attacking__line_breaks__attempted_defensive_line_breaks", "sum"),
        total_completed_defensive_line_breaks=("attacking__line_breaks__completed_defensive_line_breaks", "sum"),
    )
)

# Team-level pass completion rate
team_metrics["pass_completion_rate"] = np.where(
    team_metrics["total_passes"] > 0,
    team_metrics["total_completed_passes"] / team_metrics["total_passes"] * 100,
    np.nan,
)

# Team-level line break completion rate
team_metrics["line_break_completion_rate"] = np.where(
    team_metrics["total_attempted_line_breaks"] > 0,
    team_metrics["total_completed_line_breaks"]
    / team_metrics["total_attempted_line_breaks"]
    * 100,
    np.nan,
)

# Team-level defensive line break completion rate
team_metrics["defensive_line_break_completion_rate"] = np.where(
    team_metrics["total_attempted_defensive_line_breaks"] > 0,
    team_metrics["total_completed_defensive_line_breaks"]
    / team_metrics["total_attempted_defensive_line_breaks"]
    * 100,
    np.nan,
)

# Directness proxy 1:
# Final-third entries per 100 completed passes
team_metrics["final_third_entries_per_100_completed_passes"] = np.where(
    team_metrics["total_completed_passes"] > 0,
    team_metrics["total_final_third_entries"]
    / team_metrics["total_completed_passes"]
    * 100,
    np.nan,
)

# Directness proxy 2:
# Completed line breaks per 100 completed passes
team_metrics["completed_line_breaks_per_100_completed_passes"] = np.where(
    team_metrics["total_completed_passes"] > 0,
    team_metrics["total_completed_line_breaks"]
    / team_metrics["total_completed_passes"]
    * 100,
    np.nan,
)

# Per-match volume metrics
team_metrics["completed_passes_per_match"] = (
    team_metrics["total_completed_passes"] / team_metrics["matches_played"]
)

team_metrics["final_third_entries_per_match"] = (
    team_metrics["total_final_third_entries"] / team_metrics["matches_played"]
)

team_metrics["completed_line_breaks_per_match"] = (
    team_metrics["total_completed_line_breaks"] / team_metrics["matches_played"]
)

# Display team-level preview
team_metrics_preview_columns = [
    "team_name",
    "matches_played",
    "avg_team_possession_pct",
    "avg_in_contest_pct",
    "completed_passes_per_match",
    "final_third_entries_per_match",
    "completed_line_breaks_per_match",
    "pass_completion_rate",
    "line_break_completion_rate",
    "defensive_line_break_completion_rate",
    "final_third_entries_per_100_completed_passes",
    "completed_line_breaks_per_100_completed_passes",
]

display(
    team_metrics[team_metrics_preview_columns]
    .sort_values("avg_team_possession_pct", ascending=False)
    .round(2)
    .head(10)
)

In [ ]:
# =========================================================
# Step 6. Check team rankings for possession and directness
# =========================================================

# 1. Highest team possession
top_possession = (
    team_metrics[
        [
            "team_name",
            "matches_played",
            "avg_team_possession_pct",
            "completed_passes_per_match",
            "final_third_entries_per_100_completed_passes",
            "completed_line_breaks_per_100_completed_passes",
            "line_break_completion_rate",
        ]
    ]
    .sort_values("avg_team_possession_pct", ascending=False)
    .head(10)
    .round(2)
)

print("Top 10 teams by average team possession:")
display(top_possession)


# 2. Most direct final-third entry teams
top_final_third_directness = (
    team_metrics[
        [
            "team_name",
            "matches_played",
            "avg_team_possession_pct",
            "completed_passes_per_match",
            "final_third_entries_per_100_completed_passes",
            "final_third_entries_per_match",
        ]
    ]
    .sort_values("final_third_entries_per_100_completed_passes", ascending=False)
    .head(10)
    .round(2)
)

print("Top 10 teams by final-third entries per 100 completed passes:")
display(top_final_third_directness)


# 3. Most direct line-breaking teams
top_line_break_directness = (
    team_metrics[
        [
            "team_name",
            "matches_played",
            "avg_team_possession_pct",
            "completed_passes_per_match",
            "completed_line_breaks_per_100_completed_passes",
            "completed_line_breaks_per_match",
            "line_break_completion_rate",
        ]
    ]
    .sort_values("completed_line_breaks_per_100_completed_passes", ascending=False)
    .head(10)
    .round(2)
)

print("Top 10 teams by completed line breaks per 100 completed passes:")
display(top_line_break_directness)


# 4. Highest line break completion rate
top_line_break_completion = (
    team_metrics[
        [
            "team_name",
            "matches_played",
            "avg_team_possession_pct",
            "total_attempted_line_breaks",
            "total_completed_line_breaks",
            "line_break_completion_rate",
            "defensive_line_break_completion_rate",
        ]
    ]
    .sort_values("line_break_completion_rate", ascending=False)
    .head(10)
    .round(2)
)

print("Top 10 teams by line break completion rate:")
display(top_line_break_completion)

In [ ]:
 # =========================================================
# Step 7. Team Possession vs Final-third Directness scatter plot
# Version: manual label offsets for crowded zones
# =========================================================

plot_df = team_metrics.copy()

x_col = "avg_team_possession_pct"
y_col = "final_third_entries_per_100_completed_passes"
size_col = "matches_played"

match_color_map = {
    3: "#8b5cf6",  # purple
    4: "#3b82f6",  # blue
    5: "#10b981",  # green
    6: "#f59e0b",  # amber
    8: "#ef4444",  # red
}

plot_df["match_color"] = plot_df[size_col].map(match_color_map)

point_sizes = 105 + (plot_df[size_col] - plot_df[size_col].min()) * 60

fig, ax = plt.subplots(figsize=(26, 16), dpi=150)

# Main points
ax.scatter(
    plot_df[x_col],
    plot_df[y_col],
    c=plot_df["match_color"],
    s=point_sizes,
    alpha=0.86,
    edgecolor="#1f2937",
    linewidth=0.75,
    zorder=3,
)

# Median lines
x_median = plot_df[x_col].median()
y_median = plot_df[y_col].median()

ax.axvline(x_median, linestyle="--", color="#94a3b8", linewidth=1.1, alpha=0.70, zorder=1)
ax.axhline(y_median, linestyle="--", color="#94a3b8", linewidth=1.1, alpha=0.70, zorder=1)

ax.text(
    x_median + 0.15,
    plot_df[y_col].min() - 0.35,
    "Median team possession",
    ha="left",
    va="top",
    fontsize=11,
    color="#64748b",
)

ax.text(
    plot_df[x_col].min() + 0.30,
    y_median + 0.08,
    "Median directness",
    ha="left",
    va="bottom",
    fontsize=11,
    color="#64748b",
)

# Semifinalists highlighted
highlight_teams = ["Spain", "France", "Argentina", "England"]

# Manual label offsets for crowded teams.
# Values are (x_offset, y_offset).
label_offsets = {
    "Mexico": (-0.95, 0.22),
    "Senegal": (0.35, 0.32),
    "Switzerland": (0.35, -0.28),
    "Brazil": (-0.95, -0.34),
    "Egypt": (-0.35, -0.55),
    "Belgium": (0.35, 0.30),
    "Ecuador": (0.35, 0.18),
    "USA": (0.35, 0.48),
    "Norway": (0.35, -0.18),
    "Portugal": (0.35, 0.20),
    "Colombia": (0.35, -0.25),
    "Algeria": (0.35, 0.18),
    "Korea Republic": (0.35, -0.20),
    "Côte d'Ivoire": (-1.05, 0.18),
    "Croatia": (0.35, -0.18),
    "South Africa": (0.35, -0.25),
    "Argentina": (0.32, -0.05),
    "Spain": (0.32, 0.05),
    "France": (0.32, 0.12),
    "England": (0.32, 0.12),
}

default_x_offset = 0.18
default_y_offset = 0.08

for _, row in plot_df.iterrows():
    team = row["team_name"]
    x = row[x_col]
    y = row[y_col]

    if team in highlight_teams:
        ax.scatter(
            x,
            y,
            s=point_sizes.loc[row.name] + 320,
            facecolors="none",
            edgecolors="#dc2626",
            linewidth=3.4,
            zorder=5,
        )
        label_size = 14
        label_weight = "bold"
        label_color = "#111827"
        line_width = 1.15
        line_alpha = 0.85
    else:
        label_size = 11
        label_weight = "normal"
        label_color = "#475569"
        line_width = 0.95
        line_alpha = 0.70

    dx, dy = label_offsets.get(team, (default_x_offset, default_y_offset))
    label_x = x + dx
    label_y = y + dy

    # Leader line from point to label
    ax.plot(
        [x, label_x],
        [y, label_y],
        color="#64748b",
        linewidth=line_width,
        alpha=line_alpha,
        zorder=2,
    )

    ax.text(
        label_x,
        label_y,
        team,
        fontsize=label_size,
        fontweight=label_weight,
        color=label_color,
        ha="left",
        va="center",
        zorder=6,
    )

# Legend
for match_count in sorted(plot_df[size_col].unique()):
    ax.scatter(
        [],
        [],
        s=105 + (match_count - plot_df[size_col].min()) * 60,
        color=match_color_map.get(match_count, "#9ca3af"),
        edgecolor="#4b5563",
        alpha=0.86,
        label=f"{int(match_count)} matches",
    )

ax.scatter(
    [],
    [],
    s=420,
    facecolors="none",
    edgecolors="#dc2626",
    linewidth=3.4,
    label="Semifinalists",
)

ax.legend(
    title="Match count / highlighted teams",
    loc="lower right",
    frameon=True,
    fontsize=11,
    title_fontsize=12,
)

ax.set_title(
    "High Possession Did Not Always Mean Direct Territory Access",
    fontsize=25,
    pad=24,
)

ax.set_xlabel("Team Possession (%)", fontsize=16)
ax.set_ylabel("Final-third Directness", fontsize=16)

ax.tick_params(axis="both", labelsize=13)
ax.grid(alpha=0.16)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.margins(x=0.11, y=0.12)

footnote = (
    "Final-third Directness = final-third entries per 100 completed passes.\n"
    "Team possession uses FIFA team possession only; in-contest possession is not allocated to either team.\n"
    "Point size and color reflect matches played. Belgium vs Egypt is excluded because FIFA provides only Live Statistics for that match.\n"
    "Data source: FIFA Match Centre | Full FIFA Official Stats only."
)

fig.text(0.08, 0.025, footnote, fontsize=10.5, color="gray")

plt.tight_layout(rect=[0, 0.08, 1, 1])

output_path = OUTPUT_DIR / "21_possession_vs_final_third_directness.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")

plt.show()

print(f"Saved figure: {output_path}")

In [ ]:
# =========================================================
# Step 8. Create possession style categories
# =========================================================

style_df = team_metrics.copy()

# Use median lines as simple tournament-relative thresholds.
possession_median = style_df["avg_team_possession_pct"].median()
directness_median = style_df["final_third_entries_per_100_completed_passes"].median()

print("Median team possession:", round(possession_median, 2))
print("Median final-third directness:", round(directness_median, 2))


def classify_possession_style(row):
    possession_high = row["avg_team_possession_pct"] >= possession_median
    directness_high = row["final_third_entries_per_100_completed_passes"] >= directness_median
    
    if possession_high and directness_high:
        return "High Possession / Direct Access"
    elif possession_high and not directness_high:
        return "High Possession / Controlled Circulation"
    elif not possession_high and directness_high:
        return "Low Possession / Direct Access"
    else:
        return "Low Possession / Limited Direct Access"


style_df["possession_style"] = style_df.apply(classify_possession_style, axis=1)

style_summary = (
    style_df
    .groupby("possession_style")
    .agg(
        teams=("team_name", "count"),
        avg_possession=("avg_team_possession_pct", "mean"),
        avg_directness=("final_third_entries_per_100_completed_passes", "mean"),
        avg_matches_played=("matches_played", "mean"),
    )
    .reset_index()
    .round(2)
)

display(style_summary)

display(
    style_df[
        [
            "team_name",
            "matches_played",
            "avg_team_possession_pct",
            "final_third_entries_per_100_completed_passes",
            "possession_style",
        ]
    ]
    .sort_values(["possession_style", "avg_team_possession_pct"], ascending=[True, False])
    .round(2)
)

In [ ]:
 # =========================================================
# Step 9. Possession style map
# Final version: manual label cleanup
# =========================================================

plot_df = style_df.copy()

x_col = "avg_team_possession_pct"
y_col = "final_third_entries_per_100_completed_passes"
size_col = "matches_played"

style_color_map = {
    "High Possession / Direct Access": "#0f766e",
    "High Possession / Controlled Circulation": "#2563eb",
    "Low Possession / Direct Access": "#f59e0b",
    "Low Possession / Limited Direct Access": "#94a3b8",
}

plot_df["style_color"] = plot_df["possession_style"].map(style_color_map)

point_sizes = 100 + (plot_df[size_col] - plot_df[size_col].min()) * 60

fig, ax = plt.subplots(figsize=(24, 15), dpi=150)

# Background quadrant shading
x_min, x_max = plot_df[x_col].min() - 2, plot_df[x_col].max() + 2
y_min, y_max = plot_df[y_col].min() - 1, plot_df[y_col].max() + 1

ax.axvspan(
    possession_median,
    x_max,
    ymin=(directness_median - y_min) / (y_max - y_min),
    ymax=1,
    color="#ecfdf5",
    alpha=0.70,
    zorder=0,
)

ax.axvspan(
    possession_median,
    x_max,
    ymin=0,
    ymax=(directness_median - y_min) / (y_max - y_min),
    color="#eff6ff",
    alpha=0.75,
    zorder=0,
)

ax.axvspan(
    x_min,
    possession_median,
    ymin=(directness_median - y_min) / (y_max - y_min),
    ymax=1,
    color="#fffbeb",
    alpha=0.75,
    zorder=0,
)

ax.axvspan(
    x_min,
    possession_median,
    ymin=0,
    ymax=(directness_median - y_min) / (y_max - y_min),
    color="#f8fafc",
    alpha=0.90,
    zorder=0,
)

# Main points
for style_name, group in plot_df.groupby("possession_style"):
    ax.scatter(
        group[x_col],
        group[y_col],
        s=point_sizes.loc[group.index],
        color=style_color_map[style_name],
        alpha=0.86,
        edgecolor="#1f2937",
        linewidth=0.75,
        label=style_name,
        zorder=3,
    )

# Median lines
ax.axvline(
    possession_median,
    linestyle="--",
    color="#64748b",
    linewidth=1.2,
    alpha=0.80,
    zorder=1,
)

ax.axhline(
    directness_median,
    linestyle="--",
    color="#64748b",
    linewidth=1.2,
    alpha=0.80,
    zorder=1,
)

# Quadrant labels
ax.text(
    x_max - 0.6,
    y_max - 0.7,
    "High Possession\nDirect Access",
    fontsize=15,
    fontweight="bold",
    color="#0f766e",
    ha="right",
    va="top",
)

ax.text(
    possession_median + 0.4,
    y_min + 0.45,
    "High Possession\nControlled Circulation",
    fontsize=15,
    fontweight="bold",
    color="#2563eb",
    ha="left",
    va="bottom",
)

ax.text(
    x_min + 0.4,
    y_max - 0.7,
    "Low Possession\nDirect Access",
    fontsize=15,
    fontweight="bold",
    color="#b45309",
    ha="left",
    va="top",
)

ax.text(
    x_min + 0.4,
    y_min + 0.6,
    "Low Possession\nLimited Direct Access",
    fontsize=15,
    fontweight="bold",
    color="#64748b",
    ha="left",
    va="bottom",
)

# Highlight semifinalists
highlight_teams = ["Spain", "France", "Argentina", "England"]

for _, row in plot_df[plot_df["team_name"].isin(highlight_teams)].iterrows():
    ax.scatter(
        row[x_col],
        row[y_col],
        s=point_sizes.loc[row.name] + 260,
        facecolors="none",
        edgecolors="#dc2626",
        linewidth=3.0,
        zorder=5,
    )

# Manual label offsets for crowded zones
label_offsets = {
    # Right-side crowded zone
    "Mexico": (-1.25, 0.22),
    "Senegal": (0.95, -0.35),
    "Switzerland": (0.75, -0.55),
    "Brazil": (-1.10, -0.42),
    "Egypt": (-0.55, -0.65),
    "Belgium": (0.35, 0.34),
    "Ecuador": (0.95, -0.08),
    "USA": (0.95, 0.95),
    "Norway": (0.45, -0.28),
    "Portugal": (0.50, 0.25),
    "Colombia": (0.60, -0.55),
    "Algeria": (0.45, 0.25),
    "Korea Republic": (0.45, -0.32),

    # Median-line / center crowded zone
    "Côte d'Ivoire": (-1.15, 0.18),
    "Croatia": (0.45, -0.22),
    "South Africa": (0.45, -0.35),
    "Japan": (0.30, 0.35),
    "Panama": (0.35, -0.15),
    "Austria": (0.35, 0.25),
    "Bosnia and Herzegovina": (-1.00, -0.25),
    "Scotland": (0.30, -0.05),
    "New Zealand": (0.35, -0.15),

    # Left-side crowded zone
    "Uzbekistan": (-0.65, 0.12),
    "Cabo Verde": (0.35, -0.05),
    "Tunisia": (0.35, -0.25),
    "Ghana": (0.35, -0.15),
    "IR Iran": (0.35, 0.18),
    "Congo DR": (0.35, 0.25),
    "Czechia": (0.35, 0.20),
    "Australia": (0.35, 0.20),

    # Semifinalists
    "France": (-1.25, 0.10),
    "England": (0.90, 0.12),
    "Argentina": (0.70, -0.45),
    "Spain": (0.65, 0.05),
}

default_x_offset = 0.18
default_y_offset = 0.08

for _, row in plot_df.iterrows():
    team = row["team_name"]
    x = row[x_col]
    y = row[y_col]

    dx, dy = label_offsets.get(team, (default_x_offset, default_y_offset))
    label_x = x + dx
    label_y = y + dy

    if team in highlight_teams:
        label_size = 14
        label_weight = "bold"
        label_color = "#111827"
        line_width = 1.35
        line_alpha = 0.90
    else:
        label_size = 10.8
        label_weight = "normal"
        label_color = "#475569"
        line_width = 1.00
        line_alpha = 0.72

    ax.plot(
        [x, label_x],
        [y, label_y],
        color="#64748b",
        linewidth=line_width,
        alpha=line_alpha,
        zorder=2,
    )

    # Labels placed left of points should use right alignment.
    right_aligned_labels = [
        "France",
        "Mexico",
        "Brazil",
        "Côte d'Ivoire",
        "Bosnia and Herzegovina",
        "Uzbekistan",
    ]

    label_ha = "right" if team in right_aligned_labels else "left"

    ax.text(
        label_x,
        label_y,
        team,
        fontsize=label_size,
        fontweight=label_weight,
        color=label_color,
        ha=label_ha,
        va="center",
        zorder=6,
    )

# Legend 1: possession style colors
style_handles = []

for style_name, color in style_color_map.items():
    style_handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            label=style_name,
            markerfacecolor=color,
            markeredgecolor="#1f2937",
            markersize=9,
        )
    )

semifinalist_handle = plt.Line2D(
    [0],
    [0],
    marker="o",
    color="w",
    label="Semifinalists",
    markerfacecolor="none",
    markeredgecolor="#dc2626",
    markeredgewidth=2.5,
    markersize=11,
)

style_legend = ax.legend(
    handles=style_handles + [semifinalist_handle],
    title="Color = possession style",
    loc="lower right",
    frameon=True,
    fontsize=9.5,
    title_fontsize=10.5,
)

ax.add_artist(style_legend)

# Legend 2: match-count circle sizes
size_handles = []

for match_count in sorted(plot_df[size_col].unique()):
    size_handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            label=f"{int(match_count)} matches",
            markerfacecolor="#cbd5e1",
            markeredgecolor="#475569",
            markersize=np.sqrt(100 + (match_count - plot_df[size_col].min()) * 60) / 1.2,
        )
    )

size_legend = ax.legend(
    handles=size_handles,
    title="Size = matches played",
    loc="upper left",
    bbox_to_anchor=(1.01, 1.00),
    frameon=True,
    fontsize=9.5,
    title_fontsize=10.5,
)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

ax.set_title(
    "Possession Style Map: Control vs Direct Territory Access",
    fontsize=24,
    pad=22,
)

ax.set_xlabel("Team Possession (%)", fontsize=16)
ax.set_ylabel("Final-third Directness", fontsize=16)

ax.tick_params(axis="both", labelsize=13)
ax.grid(alpha=0.14)

for spine in ax.spines.values():
    spine.set_visible(False)

footnote = (
    "Teams are grouped relative to tournament medians, not by a machine-learning model.\n"
    "Final-third Directness = final-third entries per 100 completed passes.\n"
    "Team possession uses FIFA team possession only; in-contest possession is not allocated to either team.\n"
    "Belgium vs Egypt is excluded because FIFA provides only Live Statistics for that match.\n"
    "Data source: FIFA Match Centre | Full FIFA Official Stats only."
)

fig.text(0.08, 0.025, footnote, fontsize=10.2, color="gray")

plt.tight_layout(rect=[0, 0.085, 0.86, 1])

output_path = OUTPUT_DIR / "22_possession_style_map.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")

plt.show()

print(f"Saved figure: {output_path}")

In [ ]:
# =========================================================
# Step 10. Correlation check: Possession vs Final-third Directness
# =========================================================

from scipy.stats import pearsonr, spearmanr

# Select the two metrics used in the scatter plot.
correlation_df = (
    style_df[
        [
            "team_name",
            "matches_played",
            "avg_team_possession_pct",
            "final_third_entries_per_100_completed_passes",
        ]
    ]
    .dropna()
    .copy()
)

x = correlation_df["avg_team_possession_pct"]
y = correlation_df["final_third_entries_per_100_completed_passes"]

# Pearson correlation checks linear relationship.
pearson_r, pearson_p = pearsonr(x, y)

# Spearman correlation checks rank-based relationship.
spearman_rho, spearman_p = spearmanr(x, y)

correlation_summary = pd.DataFrame(
    {
        "relationship": ["Possession vs Final-third Directness"],
        "teams_included": [len(correlation_df)],
        "pearson_r": [pearson_r],
        "pearson_p_value": [pearson_p],
        "spearman_rho": [spearman_rho],
        "spearman_p_value": [spearman_p],
    }
).round(4)

display(correlation_summary)

print("Interpretation:")
print(
    f"Across {len(correlation_df)} teams, Pearson r = {pearson_r:.3f} "
    f"and Spearman rho = {spearman_rho:.3f}."
)
print(
    "These values are descriptive checks only. They do not prove causality."
)

In [ ]:
# =========================================================
# Step 11. Robustness check: correlation for teams with 4+ matches
# =========================================================

from scipy.stats import pearsonr, spearmanr

# Keep only teams that played at least 4 matches.
# This removes teams eliminated after exactly 3 group-stage matches.
correlation_4plus_df = (
    style_df[
        style_df["matches_played"] >= 4
    ][
        [
            "team_name",
            "matches_played",
            "avg_team_possession_pct",
            "final_third_entries_per_100_completed_passes",
        ]
    ]
    .dropna()
    .copy()
)

x_4plus = correlation_4plus_df["avg_team_possession_pct"]
y_4plus = correlation_4plus_df["final_third_entries_per_100_completed_passes"]

# Pearson correlation: linear relationship
pearson_r_4plus, pearson_p_4plus = pearsonr(x_4plus, y_4plus)

# Spearman correlation: rank-based relationship
spearman_rho_4plus, spearman_p_4plus = spearmanr(x_4plus, y_4plus)

correlation_4plus_summary = pd.DataFrame(
    {
        "sample": ["Teams with 4+ matches"],
        "teams_included": [len(correlation_4plus_df)],
        "pearson_r": [pearson_r_4plus],
        "pearson_p_value": [pearson_p_4plus],
        "spearman_rho": [spearman_rho_4plus],
        "spearman_p_value": [spearman_p_4plus],
    }
).round(4)

display(correlation_4plus_summary)

print("Interpretation:")
print(
    f"Among teams with 4+ matches, Pearson r = {pearson_r_4plus:.3f} "
    f"and Spearman rho = {spearman_rho_4plus:.3f}."
)
print(
    "This is a robustness check to see whether the all-team result changes "
    "after removing 3-match teams."
)

In [ ]:
# =========================================================
# Step 12. Compare all teams vs teams with 4+ matches
# =========================================================

correlation_comparison = pd.DataFrame(
    {
        "sample": [
            "All teams",
            "Teams with 4+ matches",
        ],
        "teams_included": [
            len(correlation_df),
            len(correlation_4plus_df),
        ],
        "pearson_r": [
            pearson_r,
            pearson_r_4plus,
        ],
        "pearson_p_value": [
            pearson_p,
            pearson_p_4plus,
        ],
        "spearman_rho": [
            spearman_rho,
            spearman_rho_4plus,
        ],
        "spearman_p_value": [
            spearman_p,
            spearman_p_4plus,
        ],
    }
).round(4)

display(correlation_comparison)